<a href="https://colab.research.google.com/github/Machine-Learning-Visao-Computacional-T1/CodigosDeAula-Python-Dados-/blob/main/Modulo2/Semana2/Semana2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf

# Pede para o TensorFlow listar quais placas de vídeo ele encontrou
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"Sucesso! O TensorFlow encontrou {len(gpus)} GPU(s) pronta(s) para uso.")
    # Mostra o nome da placa (Geralmente uma NVIDIA Tesla T4)
    !nvidia-smi
else:
    print("Nenhuma GPU encontrada. Verifique o menu Ambiente de Execução.")

Sucesso! O TensorFlow encontrou 1 GPU(s) pronta(s) para uso.
Wed Jul 29 21:24:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |           

In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.initializers import HeNormal

print("="*60)
print("PROJETO SEMANA 1: FUNDAMENTOS DE MANUTENÇÃO PREDITIVA")
print("="*60)

PROJETO SEMANA 1: FUNDAMENTOS DE MANUTENÇÃO PREDITIVA


In [3]:
# ==============================================================================
# DIA 1: TENSORES E O PIPELINE (tf.data)
# ==============================================================================
import tensorflow as tf
import numpy as np

print("\n[DIA 1] GERANDO DADOS E CRIANDO O PIPELINE...")

# 1. Gerando dados de sensores (Simulando 10.000 motores)
num_amostras = 10000
temperatura = np.random.normal(85.0, 5.0, num_amostras)
rpm = np.random.normal(1200.0, 50.0, num_amostras)
vibracao = np.random.normal(1.0, 0.3, num_amostras)
pressao = np.random.normal(98.0, 2.0, num_amostras)

# Regra simples: Alta vibração + Alta temperatura = Falha (1), senão Normal (0)
falha = np.where((vibracao > 1.6) & (temperatura > 90.0), 1.0, 0.0)

# Juntando tudo em Tensores do TensorFlow (X = Sensores, Y = Resposta)
X_dados = tf.convert_to_tensor(np.column_stack((temperatura, rpm, vibracao, pressao)), dtype=tf.float32)
Y_dados = tf.convert_to_tensor(falha, dtype=tf.float32)

# ==============================================================================
# NOVO BLOCO: EXPLORANDO A ANATOMIA DO TENSOR (Baseado no Slide 6)
# ==============================================================================
print("\n--- O QUE É UM TENSOR NA PRÁTICA? ---")

# Mostrando um Tensor 0D (Escalar)
tensor_0d = tf.constant(85.5)
print("1. Tensor 0D (Escalar - Um único valor solto):")
print(tensor_0d)

# Mostrando nosso Tensor 1D (Vetor)
print(f"\n2. Tensor 1D (Vetor Y - Nossa coluna de falhas):")
print(f"Formato (Shape): {Y_dados.shape} -> 10.000 linhas, 1 única dimensão")
print("Vendo os 5 primeiros registros:")
print(Y_dados[:5])

# Mostrando nosso Tensor 2D (Matriz)
print(f"\n3. Tensor 2D (Matriz X - Nossos 4 Sensores):")
print(f"Formato (Shape): {X_dados.shape} -> (Linhas/Amostras, Colunas/Sensores)")
print("Vendo os 3 primeiros motores:")
print(X_dados[:3])
print("-" * 60)
# ==============================================================================

# 2. O Segredo da Performance: tf.data.Dataset
tamanho_lote = 32
pipeline = tf.data.Dataset.from_tensor_slices((X_dados, Y_dados))
pipeline = pipeline.shuffle(buffer_size=1000).batch(tamanho_lote)

print(f"\n-> Base criada com {num_amostras} registros.")
print(f"-> Pipeline configurado: Entregando dados em lotes (batches) de {tamanho_lote}.")


[DIA 1] GERANDO DADOS E CRIANDO O PIPELINE...

--- O QUE É UM TENSOR NA PRÁTICA? ---
1. Tensor 0D (Escalar - Um único valor solto):
tf.Tensor(85.5, shape=(), dtype=float32)

2. Tensor 1D (Vetor Y - Nossa coluna de falhas):
Formato (Shape): (10000,) -> 10.000 linhas, 1 única dimensão
Vendo os 5 primeiros registros:
tf.Tensor([0. 0. 0. 1. 0.], shape=(5,), dtype=float32)

3. Tensor 2D (Matriz X - Nossos 4 Sensores):
Formato (Shape): (10000, 4) -> (Linhas/Amostras, Colunas/Sensores)
Vendo os 3 primeiros motores:
tf.Tensor(
[[7.95462265e+01 1.24251624e+03 1.05747700e+00 1.03772354e+02]
 [8.41996460e+01 1.18017041e+03 1.02528274e+00 1.00068954e+02]
 [8.31787033e+01 1.21870764e+03 1.09686434e+00 9.29692001e+01]], shape=(3, 4), dtype=float32)
------------------------------------------------------------

-> Base criada com 10000 registros.
-> Pipeline configurado: Entregando dados em lotes (batches) de 32.


In [4]:
# ==============================================================================
# DIA 2: A MATEMÁTICA DA REDE (FORWARD PASS)
# ==============================================================================
import tensorflow as tf

print("\n[DIA 2] COMO A REDE PENSA? (CÁLCULO MANUAL)...")

# 1. Pegando APENAS 1 lote da nossa esteira de dados (32 motores)
for lote_X, lote_Y in pipeline:

    # 2. Criando a "Memória" da Rede Neural (Pesos e Viés)
    # Temos 4 sensores (colunas) que vão se conectar a 2 Neurônios
    Pesos = tf.Variable(tf.random.normal(shape=(4, 2), stddev=0.1))
    Vies = tf.Variable(tf.zeros(shape=(2,)))

    print("\n--- A ANATOMIA DA CAMADA DENSA ---")
    print(f"A. Entrada (X): {lote_X.shape} -> 32 motores, 4 sensores")
    print(f"B. Pesos (W): {Pesos.shape} -> 4 sensores ligados a 2 neurônios")

    # 3. O Cálculo que move a Inteligência Artificial: Z = (X * Pesos) + Viés
    # tf.matmul faz a multiplicação da matriz de sensores pelos pesos
    Z = tf.matmul(lote_X, Pesos) + Vies

    print(f"\nC. O Cálculo Bruto (Z) dos 3 primeiros motores:")
    print("(Resultado antes de passar pela ativação)")
    print(Z[:3].numpy())

    # 4. A Função de Ativação ReLU
    saida_ativada = tf.nn.relu(Z)

    print("\nD. Passando pela Função ReLU (O Filtro Mágico):")
    print(saida_ativada[:3].numpy())
    print("\n-> NOTEM: A ReLU olhou o resultado bruto e transformou tudo que era negativo em ZERO!")

    break;



[DIA 2] COMO A REDE PENSA? (CÁLCULO MANUAL)...

--- A ANATOMIA DA CAMADA DENSA ---
A. Entrada (X): (32, 4) -> 32 motores, 4 sensores
B. Pesos (W): (4, 2) -> 4 sensores ligados a 2 neurônios

C. O Cálculo Bruto (Z) dos 3 primeiros motores:
(Resultado antes de passar pela ativação)
[[125.44992   76.49254 ]
 [117.99007   70.90277 ]
 [122.338066  76.24752 ]]

D. Passando pela Função ReLU (O Filtro Mágico):
[[125.44992   76.49254 ]
 [117.99007   70.90277 ]
 [122.338066  76.24752 ]]

-> NOTEM: A ReLU olhou o resultado bruto e transformou tudo que era negativo em ZERO!


In [5]:
# ==============================================================================
# DIA 3: CONSTRUINDO A ARQUITETURA NO KERAS
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from tensorflow.keras.initializers import HeNormal

print("\n[DIA 3] ESCALANDO COM KERAS E SALVANDO O PROJETO...")

# 1. Construindo a Rede Neural de forma elegante e profissional
modelo = Sequential(name="Sistema_Falhas_V1")

# Camada de Entrada: Avisamos a rede que receberemos 4 colunas (sensores)
modelo.add(Input(shape=(4,), name="Entrada_Sensores"))

# Primeira Camada Oculta: 32 Neurônios
# O kernel_initializer=HeNormal é a melhor forma de gerar os "pesos aleatórios" iniciais quando usamos ReLU
modelo.add(Dense(32, activation='relu', kernel_initializer=HeNormal(seed=42), name="Oculta_1"))

# Dropout: Desliga 20% dos neurônios (0.2) aleatoriamente durante o treino
modelo.add(Dropout(0.2, name="Evita_Decoreba"))

# Segunda Camada Oculta: Funil para 16 Neurônios
modelo.add(Dense(16, activation='relu', kernel_initializer=HeNormal(seed=42), name="Oculta_2"))

# Camada de Saída: 1 neurônio (Porque a resposta é só uma: Falhou ou não?)
# Usamos 'sigmoid' para transformar a resposta bruta em probabilidade (0% a 100%)
modelo.add(Dense(1, activation='sigmoid', name="Saida_Probabilidade"))

print("\n" + "="*60)
print("-> O RAIO-X DO NOSSO ARQUITETO (modelo.summary()):")
modelo.summary()
print("="*60)

# ==============================================================================
# PROVANDO QUE O KERAS FAZ A MATEMÁTICA PARA NÓS
# ==============================================================================
# Lembra do nosso 'lote_X' com 32 motores do Dia 2? Vamos jogar na rede pronta.
previsoes = modelo(lote_X)

print("\n-> Teste prático com 3 motores:")
print("A resposta da nossa rede recém-nascida (Sem Treinamento):")
for i in range(3):
    prob = previsoes[i].numpy()[0] * 100 # Multiplicando por 100 para virar porcentagem
    print(f"Motor {i+1}: {prob:.2f}% de chance de falha (Chute aleatório!)")

print("\n-> NOTEM: Os chutes não fazem sentido (todos beiram o mesmo valor).")
print("Por quê? Porque a rede tem a arquitetura, mas não foi TREINADA.")
print("Ela ainda está usando Pesos aleatórios!")

# 3. Salvando o progresso para a Semana 2
caminho_modelo = "modelo_destreinado.keras"
modelo.save(caminho_modelo)

print("\n" + "="*60)
print(f"SUCESSO! Modelo inicial salvo como: '{caminho_modelo}'")
print("Semana 1 concluída. Na Semana 2, vamos treinar esta rede para valer!")
print("="*60)


[DIA 3] ESCALANDO COM KERAS E SALVANDO O PROJETO...

-> O RAIO-X DO NOSSO ARQUITETO (modelo.summary()):


Model: "Sistema_Falhas_V1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Oculta_1 (Dense)                │ (None, 32)             │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Evita_Decoreba (Dropout)        │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Oculta_2 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Saida_Probabilidade (Dense)     │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 705 (2.75 KB)

 Trainable params: 705 (2.75 KB)

 Non-trainable params: 0 (0.00 B)


-> Teste prático com 3 motores:
A resposta da nossa rede recém-nascida (Sem Treinamento):
Motor 1: 100.00% de chance de falha (Chute aleatório!)
Motor 2: 100.00% de chance de falha (Chute aleatório!)
Motor 3: 100.00% de chance de falha (Chute aleatório!)

-> NOTEM: Os chutes não fazem sentido (todos beiram o mesmo valor).
Por quê? Porque a rede tem a arquitetura, mas não foi TREINADA.
Ela ainda está usando Pesos aleatórios!

SUCESSO! Modelo inicial salvo como: 'modelo_destreinado.keras'
Semana 1 concluída. Na Semana 2, vamos treinar esta rede para valer!
